In [2]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [2]:
# URL of the page with the PDFs
url = "https://www.oth-aw.de/en/studies/study-offers/study-programmes/master/artificial-intelligence-industrial-applications/structure/#documents"

# Send an HTTP GET request to fetch the page content
response = requests.get(url)

# Parse the page content with BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')

# Find all links to PDF files (e.g., <a href="some_pdf.pdf">)
pdf_links = soup.find_all('a', href=lambda href: href and href.endswith('.pdf'))


In [ ]:
# Download each PDF
for link in pdf_links:
    pdf_url = urljoin(url, link.get('href'))  # Resolve relative URL to absolute
    pdf_response = requests.get(pdf_url)
    pdf_filename = link.get_text(strip=True) + ".pdf"  # Use link text as filename
    
    with open(pdf_filename, 'wb') as f:
        f.write(pdf_response.content)
    print(f"Downloaded: {pdf_filename}")

In [4]:
'''"Bridge Course Catalogue Master Artificial Intelligence for Industrial Applications.pdf"
Course Catalogue Master Artificial Intelligence for Industrial Applications.pdf
Overview Compulsory Elective Modules for Artificial Intelligence for Industrial Applications(93 KB).pdf
Study and Examination Regulations Master Artificial Intelligence for Industrial Applications of 16.02.2023(422 KB).pdf'''

'"Bridge Course Catalogue Master Artificial Intelligence for Industrial Applications.pdf"\nCourse Catalogue Master Artificial Intelligence for Industrial Applications.pdf\nOverview Compulsory Elective Modules for Artificial Intelligence for Industrial Applications(93 KB).pdf\nStudy and Examination Regulations Master Artificial Intelligence for Industrial Applications of 16.02.2023(422 KB).pdf'

In [ ]:
def clean_text(text, is_header=False):
    if not text: return ""
    if is_header:
        return text.replace('\u2022', '').strip(" :*\n\r\t")
    text = re.sub(r'-\n\s*', '', text)
    text = text.replace('\n', ' ')
    text = re.sub(r'[¹²³⁴⁵⁶⁷⁸⁹⁰]', '', text)
    text = text.replace('\u2022', '- ')
    return " ".join(text.split())

In [23]:
def parse_combined_metadata(value):
    """Parses the 'Location Language...' combined string."""
    val_clean = value.replace('\n', ' ').strip()
    
    # Regex for: Location, Language, Duration, Frequency, [Participants]
    pattern = (
        r'(?P<Location>.*?)\s+'
        r'(?P<Language>(?:English|German|Deutsch)(?:(?:, |/)(?:English|German|Deutsch))?)\s+'
        r'(?P<Duration>.*?semester)\s+'
        r'(?P<Frequency>.*?semester)'
        r'(?:\s+(?P<Participants>.*))?$'
    )
    match = re.search(pattern, val_clean, re.IGNORECASE)
    
    if match:
        return {k: v.strip() for k, v in match.groupdict().items() if v}
    
    # Fallback: Split by double spaces
    parts = [p.strip() for p in re.split(r'\s{2,}', val_clean) if p.strip()]
    if len(parts) >= 4:
        res = {
            "Location": parts[0], "Language": parts[1],
            "Duration": parts[2], "Frequency": parts[3]
        }
        if len(parts) > 4: res["Max. Number of Participants"] = parts[4]
        return res
    return None

In [32]:
import pdfplumber
import re
import json
import sys

# Ensure UTF-8 output
if hasattr(sys.stdout, 'reconfigure'): sys.stdout.reconfigure(encoding='utf-8')
def parse_combined_metadata(value):
    """Parses the 'Location Language...' combined string."""
    val_clean = value.replace('\n', ' ').strip()
    
    # Regex for: Location, Language, Duration, Frequency, [Participants]
    pattern = (
        r'(?P<Location>.*?)\s+'
        r'(?P<Language>(?:English|German|Deutsch)(?:(?:, |/)(?:English|German|Deutsch))?)\s+'
        r'(?P<Duratrion>.*?semester)\s+'
        r'(?P<Frequency>.*?semester)'
        r'(?:\s+(?P<Participants>.*))?$'
    )
    match = re.search(pattern, val_clean, re.IGNORECASE)
    
    if match:
        return {k: v.strip() for k, v in match.groupdict().items() if v}
    
    # Fallback: Split by double spaces
    parts = [p.strip() for p in re.split(r'\s{2,}', val_clean) if p.strip()]
    if len(parts) >= 4:
        res = {
            "Location": parts[0], "Language": parts[1],
            "Duration": parts[2], "Frequency": parts[3]
        }
        if len(parts) > 4: res["Max. Number of Participants"] = parts[4]
        return res
    return None
def extract_bridge_course_catalogue(pdf_path):
    extracted_data = []
    all_lines = []

    # A. READ & LINEARIZE
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            words = page.extract_words(keep_blank_chars=True, extra_attrs=["fontname", "size"])
            words.sort(key=lambda w: (w['top'], w['x0']))
            if not words: continue
            
            current_line, lines = [words[0]], []
            for w in words[1:]:
                if abs(w['top'] - current_line[-1]['top']) < 3: current_line.append(w)
                else: lines.append(current_line); current_line = [w]
            lines.append(current_line)
            
            for line in lines:
                all_lines.append({
                    "words": line,
                    "text": " ".join([w['text'] for w in line]),
                    "page": i + 1
                })

    # B. PROCESS STREAM
    prelim_data, prelim_active, prelim_captured = {}, False, False
    prelim_topic, prelim_header_size = None, 0
    
    for idx, line_obj in enumerate(all_lines):
        text = line_obj['text']
        words = line_obj['words']
        
        # --- Preliminary Notes Handling ---
        if "preliminary notes" in text.lower() and not prelim_captured and not prelim_active:
            prelim_active = True
            prelim_header_size = words[0]['size']
            continue
        if prelim_active:
            is_end = (abs(words[0]['size'] - prelim_header_size) < 0.5 and len(text) > 5) or ("Modules" in text and words[0]['size'] > 14)
            if is_end:
                prelim_active = False
                if prelim_data:
                    extracted_data.append({"type": "preliminary_notes", "source_file": "Course Catalogue", "content": prelim_data})
                    prelim_captured = True
            else:
                if "Bold" in words[0]['fontname']:
                    topic = clean_text(text, is_header=True)
                    if topic: prelim_topic = topic; prelim_data[prelim_topic] = ""
                else:
                    if not prelim_topic: prelim_topic = "General"; prelim_data[prelim_topic] = ""
                    prelim_data[prelim_topic] += " " + clean_text(text)
            continue

        # --- Module Handling ---
        possible_ids = re.findall(r'\b[A-Z0-9]{2,5}\b', text)
        blacklist = ["ECTS", "TYPE", "KIND", "CODE", "PROF", "EXAM", "SWS", "ASPO", "NOTE", "MODUL", "MODULE", "AND", "FOR", "THE", "WITH", "ID"]
        valid_id = next((pid for pid in possible_ids if pid not in blacklist and not re.match(r'^\d+$', pid)), None)
        
        if valid_id:
            is_module = False
            for back in range(1, 3):
                if idx - back >= 0 and ("Module ID" in all_lines[idx-back]['text'] or "Modul-ID" in all_lines[idx-back]['text']):
                    is_module = True; break
            
            if is_module:
                module = {"type": "Module", "source_file": "Bridge Course Catalogue", "module_id": valid_id}
                
                # Name (Scan Up)
                name_parts = []
                for back in range(idx - back - 1, -1, -1):
                    prev = all_lines[back]
                    if "Modules" in prev['text'] and prev['words'][0]['size'] > 14: break
                    if "Preliminary notes" in prev['text']: break
                    if (prev['words'][0]['size'] > 11 or "Bold" in prev['words'][0]['fontname']):
                        if "created" in prev['text'].lower() or re.match(r'^\d+\s*$', prev['text']): continue
                        cleaned = re.sub(r'[^\w\s\-\(\)\.,&äöüÄÖÜß]', '', prev['text']).strip()
                        if cleaned: name_parts.insert(0, cleaned)
                    else:
                        if name_parts: break
                module['module_name'] = clean_text(" ".join(name_parts))

                # Metadata Window
                window = "\n".join([l['text'] for l in all_lines[idx:idx+15]])
                ects = re.search(r'(\d+)\s*ECTS', window)
                module['credits'] = int(ects.group(1)) if ects else 0
                module['semester'] = 'Summer' if re.search(r'summer', window, re.I) else 'Winter' if re.search(r'winter', window, re.I) else 'Flexible'

                # Content Scan
                learning_lines, content_lines, ass_buffer = [], [], []
                current_section, dynamic_meta, curr_meta_key, in_ass = "metadata", {}, None, False

                for fwd in range(idx + 1, len(all_lines)):
                    fwd_line = all_lines[fwd]
                    fwd_txt = fwd_line['text']
                    if ("Module ID" in fwd_txt or "Modul-ID" in fwd_txt) and len(fwd_txt) < 100: break
                    
                    if re.search(r'Method of Ass.*ment|Modulprüfungen|Prüfungsform', fwd_txt, re.I): in_ass = True; current_section = "assessment"; continue
                    if in_ass:
                        if any(x in fwd_txt.lower() for x in ["type of", "prüfungsform", "type/scope"]): continue
                        if fwd_txt.strip().startswith("*") or "refer to" in fwd_txt: in_ass = False; continue
                        if "Learning Outcomes" in fwd_txt: in_ass = False; current_section = "learning"; continue
                        if "Course Content" in fwd_txt: in_ass = False; current_section = "content"; continue
                        ass_buffer.append(fwd_txt); continue

                    if "Learning Outcomes" in fwd_txt: current_section = "learning"; continue
                    if "Course Content" in fwd_txt: current_section = "content"; continue
                    if "Teaching Material" in fwd_txt: current_section = "material"; continue

                    if current_section == "metadata":
                        is_bold = "Bold" in fwd_line['words'][0]['fontname']
                        if is_bold:
                            # Split Key/Value (Simple Bold vs Not Bold)
                            key_p, val_p = [], []
                            is_k = True
                            for w in fwd_line['words']:
                                if "Bold" in w['fontname'] and is_k: key_p.append(w['text'])
                                else: is_k = False; val_p.append(w['text'])
                            k_txt, v_txt = clean_text(" ".join(key_p), True), clean_text(" ".join(val_p))
                            
                            # HEURISTIC: If "key" is too long (likely a bold sentence) or contains commas (list), treat as value
                            if len(k_txt) > 50 or "," in k_txt:
                                if curr_meta_key:
                                    dynamic_meta[curr_meta_key] += " " + k_txt + " " + v_txt
                            else:
                                if k_txt: curr_meta_key = k_txt; dynamic_meta[curr_meta_key] = v_txt
                        elif curr_meta_key:
                            dynamic_meta[curr_meta_key] += " " + clean_text(fwd_txt)
                    elif current_section == "learning": learning_lines.append(fwd_txt)
                    elif current_section == "content": content_lines.append(fwd_txt)

                module.update(dynamic_meta)
                module['learning_outcomes'] = clean_text("\n".join(learning_lines))
                module['course_content'] = clean_text("\n".join(content_lines))
                raw_ass = "\n".join(ass_buffer).replace("Präs", "Prj").replace("Pr\u00e4s", "Prj")
                if raw_ass: module['assessment'] = {'details': clean_text(raw_ass)}

                # --- 3. APPLY THE PARSER LOGIC HERE ---
                final_module = {}
                for k, v in module.items():
                    # Identify the clumped key by checking for keywords
                    # FIXED: Added logic to handle the combined key even with typos
                    if "Location" in k and "Language" in k and "Participant" in k:  # "Participant" matches "Participants"
                        parsed = parse_combined_metadata(v)
                        if parsed: 
                            final_module.update(parsed)
                        else: 
                            final_module[k] = v
                    elif "Location" in k and "Language" in k and "Duration" in k: # Fallback for correct spelling
                         parsed = parse_combined_metadata(v)
                         if parsed: 
                            final_module.update(parsed)
                         else: 
                            final_module[k] = v
                    else:
                        final_module[k] = v
                
                extracted_data.append(final_module)

    return extracted_data


if __name__ == "__main__":
    pdf_file = "Bridge Course Catalogue Master Artificial Intelligence for Industrial Applications.pdf"
    # Ensure this file exists in your directory
    data = extract_bridge_course_catalogue(pdf_file)
    print(json.dumps(data, indent=2, ensure_ascii=False))

[
  {
    "type": "preliminary_notes",
    "source_file": "Course Catalogue",
    "content": {
      "General": " Vorbemerkungen  The bridge modules give graduates of a Bachelor's degree programme with less than 210 ECTS (but at least 180 ECTS) the opportunity to acquire the missing ECTS. They are suitable for the following master degree programme: - Artificial Intelligence for Industrial Applications   - Registration formalities: All examinations must be registered with the Students’ Office (through PRIMUSS). Add itional formalities are listed in the module descriptions. - Abbreviations: ECTS = The European Credit Transfer and Accumulation System (ECTS) is a credit point system for accreditation of course achievements. SWS = Semesterwochenstunden = Semester hours per week - Workload: According to the Bologna Process, a credit point is based on a workload of 25-30 hours. The number of hours includes the time spent at the university, the time spent preparing for and following up on cour

In [25]:
import pdfplumber
import re
import json
import sys

# Ensure UTF-8 output
if hasattr(sys.stdout, 'reconfigure'): sys.stdout.reconfigure(encoding='utf-8')

def parse_combined_metadata(value):
    """Parses the 'Location Language...' combined string."""
    val_clean = value.replace('\n', ' ').strip()
    
    # Regex for: Location, Language, Duration, Frequency, [Participants]
    pattern = (
        r'(?P<Location>.*?)\s+'
        r'(?P<Language>(?:English|German|Deutsch)(?:(?:, |/)(?:English|German|Deutsch))?)\s+'
        r'(?P<Duration>.*?semester)\s+'
        r'(?P<Frequency>.*?semester)'
        r'(?:\s+(?P<Participants>.*))?$'
    )
    match = re.search(pattern, val_clean, re.IGNORECASE)
    
    if match:
        return {k: v.strip() for k, v in match.groupdict().items() if v}
    
    # Fallback: Split by double spaces
    parts = [p.strip() for p in re.split(r'\s{2,}', val_clean) if p.strip()]
    if len(parts) >= 4:
        res = {
            "Location": parts[0], "Language": parts[1],
            "Duration": parts[2], "Frequency": parts[3]
        }
        if len(parts) > 4: res["Max. Number of Participants"] = parts[4]
        return res
    return None

def extract_course_catalogue(pdf_path):
    extracted_data = []
    all_lines = []

    # 1. READ & LINEARIZE
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            words = page.extract_words(keep_blank_chars=True, extra_attrs=["fontname", "size"])
            words.sort(key=lambda w: (w['top'], w['x0']))
            if not words: continue
            
            curr_line, lines = [words[0]], []
            for w in words[1:]:
                if abs(w['top'] - curr_line[-1]['top']) < 3: curr_line.append(w)
                else: lines.append(curr_line); curr_line = [w]
            lines.append(curr_line)
            
            for line in lines:
                all_lines.append({
                    "words": line, "text": " ".join([w['text'] for w in line]), "page": i + 1
                })

    # 2. PROCESS STREAM
    for idx, line_obj in enumerate(all_lines):
        text = line_obj['text']
        
        # Anchor: "Module ID" (4-letter code)
        possible_ids = re.findall(r'\b[A-Z]{4}\b', text)
        valid_id = next((pid for pid in possible_ids if pid not in ["ECTS", "TYPE", "KIND", "CODE", "PROF", "EXAM", "SWS", "ASPO", "NOTE"]), None)
        
        # Verify context
        if valid_id and any("Module ID" in all_lines[idx-b]['text'] for b in range(1, 3) if idx-b >= 0):
            module = {"type": "Master module", "source_file": "Master Course Catalogue", "module_id": valid_id}
            
            # 1. Name (Scan Up)
            name_parts = []
            for back in range(idx - 2, -1, -1):
                prev = all_lines[back]
                if "Classification" in prev['text'] or "Module ID" in prev['text']: continue
                if (prev['words'][0]['size'] > 11 or "Bold" in prev['words'][0]['fontname']) and "Required modules" not in prev['text']:
                    name_parts.insert(0, prev['text'])
                else:
                    if name_parts: break
            module['module_name'] = clean_text(" ".join(name_parts))

            # 2. Metadata (Window)
            window = "\n".join([l['text'] for l in all_lines[idx:idx+15]])
            ects = re.search(r'(\d+)\s*ECTS', window)
            module['credits'] = int(ects.group(1)) if ects else 0
            module['semester'] = 'Summer' if re.search(r'summer', window, re.I) else 'Winter' if re.search(r'winter', window, re.I) else 'Flexible'

            # 3. Content & Assessment (Scan Forward)
            learning, content, ass_buf = [], [], []
            curr_sec, curr_meta_key, in_ass = "metadata", None, False
            dynamic_meta = {}

            for fwd in range(idx + 1, len(all_lines)):
                fwd_line = all_lines[fwd]
                fwd_txt = fwd_line['text']
                
                if ("Module ID" in fwd_txt or "Modul-ID" in fwd_txt) and len(fwd_txt) < 100: break
                
                if re.search(r'Method of Ass.*ment|Modulprüfungen|Prüfungsform', fwd_txt, re.I): in_ass = True; curr_sec = "assessment"; continue
                if in_ass:
                    if any(x in fwd_txt.lower() for x in ["type of", "prüfungsform", "type/scope"]): continue
                    if fwd_txt.strip().startswith("*") or "refer to" in fwd_txt: in_ass = False; continue
                    if "Learning Outcomes" in fwd_txt: in_ass = False; curr_sec = "learning"; continue
                    if "Course Content" in fwd_txt: in_ass = False; curr_sec = "content"; continue
                    ass_buf.append(fwd_txt); continue

                if "Learning Outcomes" in fwd_txt: curr_sec = "learning"; continue
                if "Course Content" in fwd_txt: curr_sec = "content"; continue
                if "Teaching Material" in fwd_txt: curr_sec = "material"; continue

                if curr_sec == "metadata":
                    if "Bold" in fwd_line['words'][0]['fontname']:
                        key_p, val_p = [], []
                        is_k = True
                        for w in fwd_line['words']:
                            if "Bold" in w['fontname'] and is_k: key_p.append(w['text'])
                            else: is_k = False; val_p.append(w['text'])
                        k_txt = clean_text(" ".join(key_p), True)
                        if k_txt: curr_meta_key = k_txt; dynamic_meta[curr_meta_key] = clean_text(" ".join(val_p))
                    elif curr_meta_key: dynamic_meta[curr_meta_key] += " " + clean_text(fwd_txt)
                elif curr_sec == "learning": learning.append(fwd_txt)
                elif curr_sec == "content": content.append(fwd_txt)

            module.update(dynamic_meta)
            module['learning_outcomes'] = clean_text("\n".join(learning))
            module['course_content'] = clean_text("\n".join(content))
            if ass_buf: module['assessment'] = {'details': clean_text("\n".join(ass_buf).replace("Präs", "Prj"))}

            # Rebuild module with parsed metadata in order
            final_module = {}
            for k, v in module.items():
                if "Location" in k and "Language" in k and "Duration" in k:
                    parsed = parse_combined_metadata(v)
                    if parsed: final_module.update(parsed)
                    else: final_module[k] = v
                else:
                    final_module[k] = v
            
            extracted_data.append(final_module)

    return extracted_data

if __name__ == "__main__":
    pdf_file = "Course Catalogue Master Artificial Intelligence for Industrial Applications.pdf"
    data = extract_course_catalogue(pdf_file)
    print(json.dumps(data, indent=2, ensure_ascii=False))


[
  {
    "type": "Master module",
    "source_file": "Master Course Catalogue",
    "module_id": "DPLE",
    "module_name": "Deep Learning",
    "credits": 5,
    "semester": "Summer",
    "Location": "Amberg",
    "Language": "English",
    "Duration": "one semester",
    "Frequency": "summer semester",
    "Module Convenor  Professor / Lecturer": " Prof. Dr. Christian Bergler Prof. Dr. Christian Bergler",
    "Prerequistes": " advanced competences in computer science and mathematics",
    "Note: please also observe the preperquisites according to eximinations regulations law in the current version of the SPO.": "",
    "Usability  Teaching Methods  Workload": " Master study programmes with focus on AI Seminars with exercises Contact time: 60h Self study: 90h neu",
    "learning_outcomes": "Lernziele / Qualifikationen des Moduls After completing this module successfully, students will have the following professional, methodological and personal competences: Professional competence: S

In [14]:
#Bridge Course Catalogue to Modules

In [15]:

import pdfplumber
import re
import json
import sys

# Ensure UTF-8 output for Windows console (only if supported)
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

def extract_bridge_course_catalogue(pdf_path):
    """
    Extracts modules and preliminary notes from Course Catalogue PDFs.
    Supports both Standard and Bridge catalogues.
    """
    extracted_data = []
    all_lines = []

    # 1. READ ALL PAGES & LINEARIZE
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            words = page.extract_words(keep_blank_chars=True, extra_attrs=["fontname", "size"])
            words.sort(key=lambda w: (w['top'], w['x0']))
            
            if not words: continue
            
            # Group words into lines
            current_line = [words[0]]
            lines = []
            for w in words[1:]:
                if abs(w['top'] - current_line[-1]['top']) < 3:
                    current_line.append(w)
                else:
                    lines.append(current_line)
                    current_line = [w]
            lines.append(current_line)
            
            for line in lines:
                all_lines.append({
                    "words": line,
                    "text": " ".join([w['text'] for w in line]),
                    "page": i + 1
                })

    # 2. PROCESS STREAM
    prelim_data = {}
    prelim_active = False
    prelim_captured = False
    prelim_topic = None
    prelim_header_size = 0
    
    for idx, line_obj in enumerate(all_lines):
        text = line_obj['text']
        words = line_obj['words']
        
        # --- A. PRELIMINARY NOTES ---
        if "preliminary notes" in text.lower() and not prelim_captured and not prelim_active:
            prelim_active = True
            prelim_header_size = words[0]['size']
            continue
            
        if prelim_active:
            # Check for end of section:
            # 1. Same size as header (e.g. next big section)
            # 2. "Modules" header (Size ~17)
            is_end_of_prelim = (abs(words[0]['size'] - prelim_header_size) < 0.5 and len(text) > 5) or \
                               ("Modules" in text and words[0]['size'] > 14)
            
            if is_end_of_prelim:
                prelim_active = False
                if prelim_data:
                    extracted_data.append({"type": "preliminary_notes", "source_file": "Course Catalogue", "content": prelim_data})
                    prelim_captured = True
            else:
                is_bold = "Bold" in words[0]['fontname']
                if is_bold:
                    topic = clean_text(text, is_header=True)
                    if topic:
                        prelim_topic = topic
                        prelim_data[prelim_topic] = ""
                else:
                    # Capture content even if no topic found yet (use "General")
                    if not prelim_topic:
                        prelim_topic = "General"
                        prelim_data[prelim_topic] = ""
                    prelim_data[prelim_topic] += " " + clean_text(text) 
            continue # Skip module check if in prelim

        # --- B. MODULES ---
        # Anchor: "Module ID" (Variable length, alphanumeric)
        # Bridge Catalogue has IDs like "140P", "INT", "ROS", "PK1"
        possible_ids = re.findall(r'\b[A-Z0-9]{2,5}\b', text)
        blacklist = ["ECTS", "TYPE", "KIND", "CODE", "PROF", "EXAM", "SWS", "ASPO", "NOTE", "MODUL", "MODULE", "AND", "FOR", "THE", "WITH", "ID"]
        valid_id = next((pid for pid in possible_ids if pid not in blacklist and not re.match(r'^\d+$', pid)), None)
        
        if valid_id:
            # Verify context: "Module ID" must be in previous lines
            is_module = False
            for back in range(1, 3):
                if idx - back >= 0 and ("Module ID" in all_lines[idx-back]['text'] or "Modul-ID" in all_lines[idx-back]['text']):
                    is_module = True
                    break
            
            if is_module:
                module = {
                    "type": "Module", # Generic type, can be refined if needed
                    "source_file": "Bridge Course Catalogue",
                    "module_id": valid_id
                }
                
                # 1. Name (Scan Up)
                name_parts = []
                for back in range(idx - back - 1, -1, -1):
                    prev = all_lines[back]
                    if "Classification" in prev['text'] or "Module ID" in prev['text']: continue
                    
                    # Stop conditions for scanning up
                    if "Modules" in prev['text'] and prev['words'][0]['size'] > 14: break
                    if "Preliminary notes" in prev['text']: break
                    
                    if (prev['words'][0]['size'] > 11 or "Bold" in prev['words'][0]['fontname']) and "Required modules" not in prev['text']:
                        # Filter out artifacts (footers, page numbers)
                        if "created" in prev['text'].lower(): continue
                        if re.match(r'^\d+\s*$', prev['text']): continue
                        cleaned_line_text = re.sub(r'[^\w\s\-\(\)\.,&äöüÄÖÜß]', '', prev['text']).strip()
                        
                        # 2. Check: If the line became empty after cleaning (was just "^^"), skip it
                        if not cleaned_line_text: 
                            continue

                        name_parts.insert(0, cleaned_line_text)
                        # ---------------------------
                    else:
                        if name_parts: break
                module['module_name'] = clean_text(" ".join(name_parts))

                # 2. Metadata (Scan Immediate Window)
                window = "\n".join([l['text'] for l in all_lines[idx:idx+15]])
                ects = re.search(r'(\d+)\s*ECTS', window)
                module['credits'] = int(ects.group(1)) if ects else 0
                module['semester'] = 'Summer' if re.search(r'summer', window, re.I) else 'Winter' if re.search(r'winter', window, re.I) else 'Flexible'

                # 3. Content & Assessment (Scan Forward)
                learning_lines, content_lines, ass_buffer = [], [], []
                current_section = "metadata"
                dynamic_meta = {}
                curr_meta_key = None
                in_ass = False

                for fwd in range(idx + 1, len(all_lines)):
                    fwd_line = all_lines[fwd]
                    fwd_txt = fwd_line['text']
                    
                    # Stop at next module
                    if ("Module ID" in fwd_txt or "Modul-ID" in fwd_txt) and len(fwd_txt) < 100: break
                    
                    # Section Detection
                    if re.search(r'Method of Ass.*ment|Modulprüfungen|Prüfungsform', fwd_txt, re.IGNORECASE):
                        in_ass = True; current_section = "assessment"; continue
                    
                    if in_ass:
                        if any(x in fwd_txt.lower() for x in ["type of", "prüfungsform", "type/scope"]): continue
                        # Stop capturing if we hit the footnotes (e.g., "*1) Please refer to...")
                        if fwd_txt.strip().startswith("*") or "refer to the applicable" in fwd_txt:
                             in_ass = False 
                             continue
                        if "Learning Outcomes" in fwd_txt or "Lernziele" in fwd_txt: in_ass = False; current_section = "learning"; continue
                        if "Course Content" in fwd_txt or "Inhalte" in fwd_txt: in_ass = False; current_section = "content"; continue
                        ass_buffer.append(fwd_txt); continue

                    if "Learning Outcomes" in fwd_txt or "Lernziele" in fwd_txt: current_section = "learning"; continue
                    if "Course Content" in fwd_txt or "Inhalte" in fwd_txt: current_section = "content"; continue
                    if "Teaching Material" in fwd_txt or "Lehrmaterial" in fwd_txt: current_section = "material"; continue

                    # Capture
                    if current_section == "metadata":
                        is_bold = "Bold" in fwd_line['words'][0]['fontname']
                        if is_bold:
                            # Split Key/Value
                            key_p, val_p = [], []
                            is_k = True
                            for w in fwd_line['words']:
                                if "Bold" in w['fontname'] and is_k: key_p.append(w['text'])
                                else: is_k = False; val_p.append(w['text'])
                            k_txt = clean_text(" ".join(key_p), True)
                            v_txt = clean_text(" ".join(val_p))
                            if k_txt: curr_meta_key = k_txt; dynamic_meta[curr_meta_key] = v_txt
                        elif curr_meta_key:
                            dynamic_meta[curr_meta_key] += " " + clean_text(fwd_txt)
                    
                    elif current_section == "learning": learning_lines.append(fwd_txt)
                    elif current_section == "content": content_lines.append(fwd_txt)

                # Assemble
                module.update(dynamic_meta)
                module['learning_outcomes'] = clean_text("\n".join(learning_lines))
                module['course_content'] = clean_text("\n".join(content_lines))
                
                raw_ass = "\n".join(ass_buffer).replace("Präs", "Prj").replace("Pr\u00e4s", "Prj")
                if raw_ass: module['assessment'] = {'details': clean_text(raw_ass)}
                
                extracted_data.append(module)

    return extracted_data



# Run extraction
pdf_file = "Bridge Course Catalogue Master Artificial Intelligence for Industrial Applications.pdf"
data = extract_bridge_course_catalogue(pdf_file)
print(json.dumps(data, indent=2, ensure_ascii=False))

[
  {
    "type": "preliminary_notes",
    "source_file": "Course Catalogue",
    "content": {
      "General": " Vorbemerkungen  The bridge modules give graduates of a Bachelor's degree programme with less than 210 ECTS (but at least 180 ECTS) the opportunity to acquire the missing ECTS. They are suitable for the following master degree programme: - Artificial Intelligence for Industrial Applications   - Registration formalities: All examinations must be registered with the Students’ Office (through PRIMUSS). Add itional formalities are listed in the module descriptions. - Abbreviations: ECTS = The European Credit Transfer and Accumulation System (ECTS) is a credit point system for accreditation of course achievements. SWS = Semesterwochenstunden = Semester hours per week - Workload: According to the Bologna Process, a credit point is based on a workload of 25-30 hours. The number of hours includes the time spent at the university, the time spent preparing for and following up on cour

In [ ]:
#STDUY PDF

In [9]:
import pdfplumber
import re
import json
import sys

# Ensure UTF-8 output for Windows console (only if supported)
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

def clean_text(text):
        if not text: return ""
        
        # 1. Fix Hyphenation: "indepen-\ndently" -> "independently"
        # Look for a hyphen followed by a newline (and optional spaces)
        text = re.sub(r'-\n\s*', '', text)
        
        # 2. Fix Line Breaks: "Examination\nRegulations" -> "Examination Regulations"
        # Replace remaining newlines with a space
        text = text.replace('\n', ' ')
        
        # 3. Remove Superscripts (¹ ² ³)
        text = re.sub(r'[¹²³⁴⁵⁶⁷⁸⁹⁰]', '', text)
        
        # 4. Remove Bullet Points
        text = text.replace('\u2022', '- ')
        
        # 5. Collapse multiple spaces: "Examination  Regulations" -> "Examination Regulations"
        text = re.sub(r'\s+', ' ', text)
        
        return text.strip()

def extract_regulations(pdf_path):
    """
    Extracts sections from Study and Examination Regulations PDF.
    """
    extracted_data = []
    

    with pdfplumber.open(pdf_path) as pdf:
        all_lines = []
        
        # 1. READ ALL PAGES & LINEARIZE
        # MODIFICATION: Only process the first 8 pages
        for i, page in enumerate(pdf.pages[:8]):
            words = page.extract_words(keep_blank_chars=True, extra_attrs=["fontname", "size"])
            words.sort(key=lambda w: (w['top'], w['x0']))
            
            if not words: continue
            
            # Group words into lines
            current_line = [words[0]]
            lines = []
            for w in words[1:]:
                if abs(w['top'] - current_line[-1]['top']) < 3:
                    current_line.append(w)
                else:
                    lines.append(current_line)
                    current_line = [w]
            lines.append(current_line)
            
            for line in lines:
                all_lines.append({
                    "words": line,
                    "text": " ".join([w['text'] for w in line]),
                    "page": i + 1
                })

    # 2. PROCESS STREAM
    current_section = None
    last_line_was_header = False
    
    for idx, line_obj in enumerate(all_lines):
        text = line_obj['text']
        words = line_obj['words']
        
        # HEADER DETECTION
        # Rule: Bold AND (Size 11.04 OR Size 12.00)
        is_bold = "Bold" in words[0]['fontname']
        font_size = words[0]['size']
        is_header_size = abs(font_size - 11.04) < 0.1 or abs(font_size - 12.00) < 0.1
        is_header = is_bold and is_header_size

        clean_line = clean_text(text)

        if is_header and clean_line:
            # MODIFICATION: Stop extraction if we reach Appendix 1
            if "Appendix 1" in clean_line:
                break
                
            if last_line_was_header and current_section:
                # Continuation of the previous title
                current_section["section_title"] += " " + clean_line
            else:
                # New Section
                current_section = {
                    "type": "regulation_section",
                    # MODIFICATION: Add source_file field
                    "source_file": "Study and Examination Regulations",
                    "section_title": clean_line,
                    "content": ""
                }
                extracted_data.append(current_section)
            
            last_line_was_header = True
            
        else:
            # CONTENT
            if current_section:
                if current_section["content"]:
                    current_section["content"] += " " + clean_text(text)
                else:
                    current_section["content"] = clean_text(text)
            
            last_line_was_header = False

    return extracted_data

# Run extraction
pdf_file = "Study and Examination Regulations Master Artificial Intelligence for Industrial Applications of 16.02.2023(347 KB).pdf"
data = extract_regulations(pdf_file)
print(json.dumps(data, indent=2, ensure_ascii=False))


[
  {
    "type": "regulation_section",
    "source_file": "Study and Examination Regulations",
    "section_title": "translated on the 21th of april 2023 from original document published on 16th of february 2023",
    "content": "(for these study and examination regulations the General Study and Examination Regulations (ASPO) of the OTH Amberg-Weiden dated 27.05.2020)  Based on Art. 13 Para. 1 Sentence 2, Art. 43 Para. 5, Art. 58 Para. 1 Sentence 1, Art. 61 Para. 2 Sentence 1 and Para. 8 of the Bavarian Higher Education Act of May 23, 2006 (GVBl p. 245, BayRS 2210-1-1-WK), as amended, the OTH Amberg-Weiden issues the following statutes:  "
  },
  {
    "type": "regulation_section",
    "source_file": "Study and Examination Regulations",
    "section_title": "§ 1 Purpose of the Study and Examination Regulations",
    "content": "These Study and Examination Regulations serve to fill out and supplement the Framework Examination Regulations for Universities of Applied Sciences in Bavaria 

In [ ]:
#Electives Extraction

In [10]:
import pdfplumber
import json
import sys

# Ensure UTF-8 output for Windows console to avoid encoding errors
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

def extract_electives_dynamic(pdf_path):
    extracted_data = []
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                tables = page.extract_tables()
                
                for table in tables:
                    if len(table) < 2: continue
                    
                    # --- STEP 1: FIND COLUMN INDICES ---
                    # Default Indices (Fallback)
                    idxs = {
                        "sose": 0,
                        "title_eng": 2,
                        "language": 3,
                        "professor": 4,
                        "ects": 7,
                        "mai_filter": 11
                    }

                    # Analyze the first row (Header) to find dynamic column positions
                    header_row = table[0]
                    for i, cell in enumerate(header_row):
                        if cell:
                            norm = cell.replace('\n', ' ').lower()
                            # Search for the specific "MAI" column header
                            if "artificial" in norm and "intelligence" in norm and "mai" in norm:
                                idxs["mai_filter"] = i
                            # You can add more dynamic column detection here if needed
                    
                    # --- STEP 2: EXTRACT DATA ---
                    # Iterate through the rest of the rows (skipping the header)
                    for row in table[1:]:
                        # Skip rows that are too short to contain the MAI column
                        if len(row) <= idxs["mai_filter"]: continue

                        # 1. CHECK THE FILTER (The "WPM" check)
                        mai_cell = row[idxs["mai_filter"]]
                        
                        # Only proceed if "WPM" is in the target cell
                        if mai_cell and "WPM" in mai_cell:
                            
                            # Helper to safely get data from a column index
                            def get_col(idx):
                                return row[idx] if idx is not None and len(row) > idx and row[idx] else ""

                            raw_sose = get_col(idxs["sose"])
                            title = get_col(idxs["title_eng"])
                            language = get_col(idxs["language"])
                            professor = get_col(idxs["professor"])
                            ects = get_col(idxs["ects"])

                            # 2. APPLY LOGIC (Winter/Summer)
                            # Default to Winter; change to Summer if "yes"/"ja" is found in SoSe column
                            availability = "Winter"
                            if raw_sose and ("yes" in raw_sose.lower() or "ja" in raw_sose.lower()):
                                availability = "Summer"

                            # 3. CLEANUP & BUILD ENTRY
                            entry = {
                                "type": "Elective Module",
                                "module_name": title.replace('\n', ' ').strip(),
                                "availability": availability,
                                "language": language.replace('\n', ' ').strip(),
                                "ects": ects.replace('\n', '').strip(),
                                "professor": professor.replace('\n', ' ').strip()
                            }
                            extracted_data.append(entry)

    except Exception as e:
        # Print error to stderr so it doesn't break JSON output in stdout
        sys.stderr.write(f"Error processing PDF '{pdf_path}': {e}\n")
        return []

    return extracted_data


# Define the specific PDF file to process
pdf_file = "Overview Compulsory Elective Modules for Artificial Intelligence for Industrial Applications(93 KB).pdf"

# Run the extraction
# Note: Ensure the PDF file exists in the same directory or provide full path
all_data = extract_electives_dynamic(pdf_file)

# Print the result as formatted JSON
print(json.dumps(all_data, indent=2, ensure_ascii=False))

[
  {
    "type": "Elective Module",
    "module_name": "Selected Topics of Augmented & Virtual Reality",
    "availability": "Winter",
    "language": "German",
    "ects": "5",
    "professor": "Prof. Gerald Pirkl"
  },
  {
    "type": "Elective Module",
    "module_name": "Selected Topics of Artificial Intelligence",
    "availability": "Winter",
    "language": "German / English",
    "ects": "5",
    "professor": "Prof. Tatyana Ivanovska / Prof. Dominikus Heckmann"
  },
  {
    "type": "Elective Module",
    "module_name": "Embedded Intelligence",
    "availability": "Summer",
    "language": "German / English",
    "ects": "5",
    "professor": "Prof. Gerald Pirkl"
  },
  {
    "type": "Elective Module",
    "module_name": "AI Privacy & Security",
    "availability": "Summer",
    "language": "English",
    "ects": "5",
    "professor": "Prof. Patrick Levi"
  },
  {
    "type": "Elective Module",
    "module_name": "Energy Management with AI Methods",
    "availability": "Winter"